**Environment Setup and Dependencies**

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
!pip install ultralytics roboflow -q

CUDA Available: True


**Authenticate and Download Data (Roboflow)**

In [ ]:
from roboflow import Roboflow


from roboflow import Roboflow
rf = Roboflow(api_key="FPluF3Ykm3rv6jAmEP5h")
project = rf.workspace("dilsara-vptum").project("night-driving-project")
version = project.version(3)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Night-Driving-Project-3 in yolov8:: 100%|██████████| 812/812 [00:00<00:00, 3587.53it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Training the Models **(YOLOv8s)**

In [ ]:
from ultralytics import YOLO

# Load the YOLOv8 Small model (pre-trained on COCO)
model = YOLO('yolov8s.pt')

# Train the model
results = model.train(
    data=f"{dataset.location}/data.yaml",  # Path to dataset config
    epochs=50,                             # Number of training passes
    imgsz=640,                             # Image resolution
    batch=16,                              # Batch size (adjust if you run out of memory)
    name='headlight_adjuster_v1'           # Name of this training run
)

Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Night-Driving-Project-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=headlight_adjuster_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, persp

Training the Models **(YOLOv8m)**

In [ ]:
from ultralytics import YOLO

# Load the YOLOv8 medium model (pre-trained on COCO)
model = YOLO('yolov8m.pt')

# Train the model
results = model.train(
    data=f"{dataset.location}/data.yaml",  # Path to dataset config
    epochs=50,                             # Number of training passes
    imgsz=640,                             # Image resolution
    batch=16,                              # Batch size (adjust if you run out of memory)
    name='headlight_adjuster_v1'           # Name of this training run
)

Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Night-Driving-Project-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=headlight_adjuster_v12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pers

Executive Summary
The training phase successfully produced two models (YOLOv8s and YOLOv8m). As per the primary objective, Vehicle Detection is performing well, with the models reliably identifying oncoming cars to trigger headlight dimming.

However, the secondary safety objective; Pedestrian Detection, remains the biggest area for improvement in future iterations.

**Strengths of the model:**

**-Robust Vehicle Detection**: The system is achieving nearly 80% mAP and ~70% Recall for vehicles. This means the core functionality—dimming lights for oncoming traffic—will work reliably in the vast majority of cases.

**-High Precision (Medium Model)**: The Medium model achieved a Precision of 0.81 (81%) for vehicles. This is excellent; it means when the model says "there is a car," it is almost certainly correct, minimizing false headlight dimming.

**-Real-Time Readiness**: Even the larger Medium model runs at 12.2ms inference time. This leaves plenty of computational headroom for your post-processing logic (tracking, headlight control signals) on edge devices like a Jetson Nano or Orin.

**Areas of improvements to be addressed in future iterations:**

While vehicle detection is solid, the Pedestrian performance remains a critical safety gap. A Recall of ~37% means the system misses roughly 6 out of 10 pedestrians. This is an area to be addressed in future iterations.